# British Airways Customer Booking Prediction

---

| | |
|:---|:---|
| **Author** | Rutuja |
| **Date** | April 2026 |
| **Domain** | Aviation / Airline |
| **Language** | Python 3 |
| **Libraries** | Pandas, NumPy, Matplotlib, scikit-learn |
| **Model** | Random Forest Classifier |

---

---

## Problem Statement

In the airline industry, a significant proportion of customers who initiate the booking process do not complete their purchase. Understanding the factors that influence booking completion is critical for airlines to optimize their conversion strategies, reduce revenue leakage, and improve customer experience. This project aims to build a predictive classification model using historical customer booking data to determine whether a customer will complete a flight booking. The model leverages customer demographics, travel characteristics, and service preferences to identify key drivers of booking completion and to provide actionable insights for business decision-making.

---

## Objective

- Analyse customer booking behaviour using exploratory data analysis.
- Build a Random Forest classification model to predict booking completion.
- Evaluate model performance using accuracy, ROC-AUC, and classification metrics.
- Identify the most influential features driving booking completion.

---

## Step 1: Import Required Libraries

The following libraries are used throughout this analysis:
- **pandas** and **numpy** for data handling and numerical operations.
- **matplotlib** for data visualisation.
- **scikit-learn** for preprocessing, model training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## Step 2: Load the Dataset

The dataset contains customer-level booking behaviour with approximately 50,000 records. The target variable is `booking_complete`, where:
- **1** indicates the booking was completed.
- **0** indicates the booking was not completed.

In [ ]:
df = pd.read_csv("Data/customer_booking.csv", encoding='latin1')
df.head()

### Interpretation

The sample data represents customer flight booking attempts for the AKL-DEL route, predominantly round-trip journeys booked through the internet. Customers vary in the number of passengers, booking lead time, and length of stay. Some customers opted for additional services such as extra baggage, preferred seats, or in-flight meals, while others did not. Despite early booking in many cases, these sample records show that the bookings were not completed (`booking_complete = 0`), indicating that factors beyond early planning -- such as preferences, timing, or customer origin -- may influence final booking decisions.

## Step 3: Exploratory Data Analysis

### 3.1 Dataset Structure

Examining data types, missing values, and the overall structure of the dataset.

In [ ]:
df.info()

### Interpretation

The dataset contains 50,000 records and 14 columns with no missing values. It includes both numerical and categorical features related to customer bookings. The target variable `booking_complete` is a binary column, making the data clean and ready for modelling after encoding the categorical variables.

### 3.2 Inspect Categorical Values

Checking the unique values in the `flight_day` column to understand its categorical nature.

In [ ]:
df["flight_day"].unique()

### Interpretation

The `flight_day` column contains categorical values representing the days of the week (Monday to Sunday). Since these are non-numeric, they need to be converted into numerical form before being used in a machine learning model.

### 3.3 Encode Flight Day

Mapping day names to numerical values (1 = Monday through 7 = Sunday) for model compatibility.

In [ ]:
mapping = {
    "Mon": 1,
    "Tue": 2,
    "Wed": 3,
    "Thu": 4,
    "Fri": 5,
    "Sat": 6,
    "Sun": 7
}

df["flight_day"] = df["flight_day"].map(mapping)
df["flight_day"].unique()

### Interpretation

The `flight_day` column has been successfully converted from categorical day names to numerical values (1 through 7), representing days from Monday to Sunday. The column is now suitable for use in the classification model.

### 3.4 Summary Statistics

Generating descriptive statistics for all numerical variables in the dataset.

In [ ]:
df.describe()

### Interpretation

The summary statistics reveal the following patterns:

- **Passengers:** Most bookings involve 1 to 2 passengers.
- **Purchase Lead Time:** The average purchase lead time is approximately 85 days, indicating many customers book well in advance.
- **Length of Stay:** The typical trip duration is around 23 days.
- **Flight Timing:** Flights are most commonly scheduled around mid-day.
- **Service Add-ons:** Extra baggage, preferred seats, and in-flight meals are selected by a significant portion of customers.
- **Target Variable:** The low mean value of `booking_complete` confirms that only a small percentage of bookings are completed, indicating class imbalance in the dataset.

### 3.5 Target Variable Distribution

Examining the proportion of completed versus non-completed bookings to assess class balance.

In [ ]:
df['booking_complete'].value_counts(normalize=True)

### Interpretation

The target variable `booking_complete` is highly imbalanced, with approximately 85% of records representing incomplete bookings and only around 15% representing completed bookings. This imbalance must be addressed during model training to prevent the classifier from being biased towards the majority class.

## Step 4: Data Preparation for Modelling

Separating the input features from the target variable and applying one-hot encoding to convert all remaining categorical variables into numerical format.

In [ ]:
X = df.drop('booking_complete', axis=1)
y = df['booking_complete']

# One-hot encode categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

print(f"Feature matrix shape: {X_encoded.shape}")
print(f"Target variable shape: {y.shape}")

## Step 5: Train-Test Split

Splitting the data into training (80%) and testing (20%) sets. Stratification is applied to ensure both sets maintain the same class distribution as the original dataset.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size:  {X_test.shape[0]}")

## Step 6: Model Training -- Random Forest Classifier

A Random Forest Classifier with 200 decision trees is trained to predict booking completion. The `class_weight='balanced'` parameter is applied to handle the class imbalance by assigning higher weights to the minority class during training. The `random_state=42` parameter ensures reproducible results.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

### Interpretation

The Random Forest model has been successfully trained on the training data. Using 200 estimators provides a robust ensemble of decision trees for classification. The balanced class weights ensure that the model does not disproportionately favour the majority class (non-completed bookings) during training.

## Step 7: Model Evaluation

Evaluating the trained model on the test set using multiple performance metrics:
- **Accuracy:** Overall proportion of correct predictions.
- **ROC-AUC Score:** Measures the model's ability to discriminate between classes.
- **Classification Report:** Provides precision, recall, and F1-score for each class.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### Interpretation

The model achieves an accuracy of approximately 85%, though this figure is primarily driven by correctly predicting non-completed bookings (the majority class). The ROC-AUC score of approximately 0.79 indicates good overall discrimination ability between the two classes.

However, the recall for completed bookings (`booking_complete = 1`) is notably low, indicating that while the model is effective at identifying non-bookings, it struggles to correctly capture customers who actually complete a booking. This is a common challenge when working with imbalanced datasets.

## Step 8: Cross-Validation

Performing 5-fold cross-validation using ROC-AUC as the scoring metric to evaluate the stability and generalisability of the model across different subsets of the data.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf, X_encoded, y,
    cv=5,
    scoring='roc_auc'
)

print("Cross-Validation AUC Scores:", np.round(cv_scores, 4))
print("Mean CV AUC:", round(cv_scores.mean(), 4))
print("Std CV AUC: ", round(cv_scores.std(), 4))

### Interpretation

The cross-validation results show high variation in ROC-AUC scores across different folds, with a low mean AUC of approximately 0.48. This indicates that the model's performance is unstable across subsets of the data and may not generalise well to unseen data.

The inconsistency suggests the presence of class imbalance effects and potential data complexity. These results highlight the need for further model tuning, feature engineering, or exploration of alternative modelling approaches such as gradient boosting, SMOTE resampling, or hyperparameter optimisation.

## Step 9: Feature Importance Analysis

Identifying the top 15 features that contribute most to the model's prediction of booking completion. Feature importance is derived from the Random Forest model based on the mean decrease in impurity across all decision trees.

In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)[-15:]

plt.figure(figsize=(10, 7))
plt.barh(range(len(indices)), importances[indices], color='#2c7bb6', edgecolor='#1a5276')
plt.yticks(range(len(indices)), X_encoded.columns[indices], fontsize=11)
plt.xlabel("Feature Importance", fontsize=12)
plt.title("Top 15 Features Influencing Booking Completion", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Interpretation

The feature importance analysis reveals the following key findings:

- **purchase_lead** is the most influential factor in predicting booking completion, indicating that customers who plan earlier exhibit distinctly different behaviour compared to last-minute bookers.
- **length_of_stay**, **flight_hour**, and **flight_day** are also highly ranked, suggesting that travel timing and trip duration strongly affect booking decisions.
- **booking_origin**, **flight_duration**, and **num_passengers** contribute meaningfully to the predictions.
- Add-on services such as **extra baggage**, **preferred seats**, and **in-flight meals** have a smaller but noticeable impact.

Overall, booking completion is driven primarily by planning behaviour and travel characteristics rather than optional service selections alone.

---

## Conclusion

This analysis demonstrates the application of a Random Forest Classifier to predict customer booking completion for British Airways. The key takeaways are:

1. **Purchase lead time** is the strongest predictor of whether a customer will complete a booking.
2. The dataset exhibits significant **class imbalance** (85% non-completed vs. 15% completed), which directly impacts model performance on the minority class.
3. The model achieves reasonable accuracy (~85%) and ROC-AUC (~0.79), but recall for completed bookings remains low.
4. Cross-validation results indicate that the model may require further tuning to improve generalisability.

### Recommendations for Future Work

- Apply resampling techniques (e.g., SMOTE) to better address class imbalance.
- Experiment with gradient boosting models (e.g., XGBoost, LightGBM) for potentially improved performance.
- Conduct hyperparameter tuning using grid search or randomised search.
- Explore additional feature engineering to capture interaction effects between variables.

---

**Author:** Rutuja1423